In [10]:
from pathlib import Path
import sys
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd().resolve().parents[1]
sys.path.append(str(PROJECT_ROOT))

from src.datasets.multimodal_feature_dataset import MultiModalFeatureDataset
from src.models.multimodal.multimodal_pvd_classifier import MultimodalPVDClassifier

In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"

Config

In [12]:
IMAGE_FEATURE_PATH = PROJECT_ROOT / "data" / "features" / "encoded_image" / "train_original_image_features_convnext_cbam.npy"
TEXT_FEATURE_PATH = PROJECT_ROOT / "data" / "features" / "encoded_text" / "train_text_embeddings_clip_vitl14.npy"
METADATA_PATH = PROJECT_ROOT / "data" / "features" / "encoded_image" / "train_original_image_features_convnext_cbam_metadata.json"

NUM_CLASSES = 28
BATCH_SIZE = 32
EPOCHS = 20
LR = 1e-4

Dataset

In [13]:
dataset = MultiModalFeatureDataset(
    image_feature_path=IMAGE_FEATURE_PATH,
    text_feature_path=TEXT_FEATURE_PATH,
    metadata_path=METADATA_PATH
)

loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

Model

In [14]:
model = MultimodalPVDClassifier(
    image_input_dim=1024,
    text_input_dim=768,
    proj_dim=512,
    proj_hidden_dim=768,
    pvd_hidden_dim=768,
    cls_hidden_dim=1024,
    num_classes=NUM_CLASSES,
    dropout=0.3,
    normalize_projection=False
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

Train loop

In [15]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for batch in loader:
        image_feat = batch["image_feat"].to(device)
        text_feat = batch["text_feat"].to(device)
        labels = batch["label"].to(device)

        logits = model(image_feat, text_feat)   # [B, num_classes]
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)

        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = total_loss / total
    epoch_acc = correct / total

    print(f"Epoch [{epoch+1}/{EPOCHS}] - Loss: {epoch_loss:.4f} - Acc: {epoch_acc:.4f}")

Epoch [1/20] - Loss: 2.6794 - Acc: 0.2791
Epoch [2/20] - Loss: 1.3778 - Acc: 0.5758
Epoch [3/20] - Loss: 0.9765 - Acc: 0.6824
Epoch [4/20] - Loss: 0.8184 - Acc: 0.7217
Epoch [5/20] - Loss: 0.6881 - Acc: 0.7611
Epoch [6/20] - Loss: 0.5793 - Acc: 0.8078
Epoch [7/20] - Loss: 0.5107 - Acc: 0.8262
Epoch [8/20] - Loss: 0.4502 - Acc: 0.8540
Epoch [9/20] - Loss: 0.3793 - Acc: 0.8716
Epoch [10/20] - Loss: 0.3293 - Acc: 0.8883
Epoch [11/20] - Loss: 0.2676 - Acc: 0.9165
Epoch [12/20] - Loss: 0.2488 - Acc: 0.9161
Epoch [13/20] - Loss: 0.2152 - Acc: 0.9268
Epoch [14/20] - Loss: 0.1773 - Acc: 0.9478
Epoch [15/20] - Loss: 0.1532 - Acc: 0.9521
Epoch [16/20] - Loss: 0.1332 - Acc: 0.9615
Epoch [17/20] - Loss: 0.1087 - Acc: 0.9688
Epoch [18/20] - Loss: 0.1030 - Acc: 0.9653
Epoch [19/20] - Loss: 0.0982 - Acc: 0.9688
Epoch [20/20] - Loss: 0.0809 - Acc: 0.9739


In [16]:
SAVE_DIR = PROJECT_ROOT / "archive" / "multimodal"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = SAVE_DIR / "multimodal_pvd_classifier_state_dict.pth"

torch.save(model.state_dict(), MODEL_PATH)
print("Saved model to:", MODEL_PATH)

Saved model to: /media/data3/users/luongdth/MulCo-PlantNet/archive/multimodal/multimodal_pvd_classifier_state_dict.pth
